# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jessica245818/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My provisional **Refresh / Content Opportunity Scoring** lane is primarily a **ranking/scoring task**: “Which content items should an editor review first?” A supervised classifier may estimate the probability of a later observed decline, but that probability is an intermediate score—not the product. The useful output is an ordered, capacity-aware queue with confidence labels and observable reason codes. A content editor uses the top of that queue to choose pages for refresh, metadata review, expansion, protection, or monitoring; the system never edits or prunes automatically.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Find the repository root in both Colab and local runs.
repo_root = Path.cwd()
while not (repo_root / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Could not find the starter dataset.")
    repo_root = repo_root.parent

df = pd.read_csv(repo_root / "data" / "raw" / "content_refresh_anonymized.csv")
lane_slice = df[df["impressions_90d"].gt(0) & df["content_age_days"].ge(90)].copy()
print(f"Task: rank {len(lane_slice):,} eligible content items for human review.")


Task: rank 30,000 eligible content items for human review.


## 2. Target or proxy

For this starter exercise I use `is_declining_proxy = (trend_direction == "down")`. This is a **teaching proxy**, not an ideal capstone target: `trend_direction` is derived from current-window `trend_pct`, so both fields encode the answer and must be excluded from features. The stronger warehouse target will be a later observed outcome such as `declined_next_30d`, defined after a decision date from a future 30-day window, with features built only from the prior 90 days. The exact decline magnitude, persistence, and minimum-volume policy will be written in the data contract before modeling. The model may estimate this outcome, while the final ranking can also consider exposure and review capacity.

In [2]:
# Sketch the starter proxy column. It is an outcome for evaluation, never an input feature.
lane_slice["is_declining_proxy"] = lane_slice["trend_direction"].eq("down").astype("int8")
target_preview = lane_slice[["content_id", "trend_direction", "is_declining_proxy"]].head()
display(target_preview)
print("Proxy positives:", f"{lane_slice['is_declining_proxy'].sum():,}")
print("Proxy positive rate:", f"{lane_slice['is_declining_proxy'].mean():.1%}")


,content_id,trend_direction,is_declining_proxy
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1


Proxy positives: 16,262
Proxy positive rate: 54.2%


## 3. Success metric

The primary metric is **Precision@50**, because 50 items is the provisional number an editor can review in one cycle. It asks what share of the first 50 ranked items have the observed target outcome. “Good” is defined before training as at least a **10 percentage-point absolute improvement over the transparent fixed-rule queue**, with both methods evaluated on the same unseen clients and later time period. On this current-window teaching slice the rule below reaches 0.36, so 0.46 is the illustrative hurdle—not a final performance claim. The capstone hurdle will be recomputed on the leakage-safe future outcome; I will also inspect recall/coverage and the top cases by hand because Precision@50 alone cannot show every failure.

In [3]:
def precision_at_k(labels, scores, k=50):
    order = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return float(np.asarray(labels)[order].mean())

# Transparent teaching baseline: visible + stale + position opportunity + depth gap.
visibility = np.log1p(lane_slice["impressions_90d"]).rank(pct=True)
freshness = lane_slice["days_since_last_update"].rank(pct=True)
position = (
    (1 - (lane_slice["avg_position"].clip(1, 50) - 1) / 49)
    * visibility
    * lane_slice["avg_position"].gt(0)
)
depth_gap = (1 - lane_slice["word_count"].rank(pct=True)) * visibility
lane_slice["fixed_rule_score"] = (
    0.40 * visibility + 0.30 * freshness + 0.25 * position + 0.05 * depth_gap
)

rule_p50 = precision_at_k(
    lane_slice["is_declining_proxy"], lane_slice["fixed_rule_score"], k=50
)
print(f"Teaching-slice fixed-rule Precision@50: {rule_p50:.2f}")
print(f"Illustrative ML hurdle (+0.10 absolute): {rule_p50 + 0.10:.2f}")
print("Caveat: this is an in-snapshot framing check, not final validation.")


Teaching-slice fixed-rule Precision@50: 0.36
Illustrative ML hurdle (+0.10 absolute): 0.46
Caveat: this is an in-snapshot framing check, not final validation.


## 4. The unit of analysis, as a real dataframe

**One row is one pseudonymized content item at the starter snapshot date.** IDs are shown only to make the grain testable; they are grouping keys, never model features. The lane slice keeps items with observed impressions and at least 90 days of content age. In the warehouse version, daily facts will be aggregated into one row per `content_hash_id` and decision date, using a prior feature window and a non-overlapping future target window.

In [4]:
unit_columns = [
    "content_id", "client_id", "impressions_90d", "clicks_90d", "avg_position",
    "ctr", "content_age_days", "days_since_last_update", "engagement_rate",
    "scroll_rate", "is_declining_proxy",
]
unit_df = lane_slice[unit_columns].copy()

assert unit_df["content_id"].is_unique
assert len(unit_df) == unit_df["content_id"].nunique()
print("Shape:", unit_df.shape)
print("Grain check: one row per content_id =", unit_df["content_id"].is_unique)
display(unit_df.head())


Shape: (30000, 11)
Grain check: one row per content_id = True


,content_id,client_id,impressions_90d,clicks_90d,avg_position,ctr,content_age_days,days_since_last_update,engagement_rate,scroll_rate,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,3803,29,10.6,0.76,187,20,5.88,4.55,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,20.3,0.05,445,25,0.00,10.00,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,0.09,141,20,0.00,28.57,1
3,content_331d6c4de07b,client_19581e27de,11751,58,6.2,0.49,463,22,1.28,3.45,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,44.0,0.13,263,14,0.00,24.29,1


## 5. Why ML beats a fixed rule here

A fixed rule is mandatory as the baseline, but it assigns the same importance to signals across content types, clients, positions, and evidence levels. Refresh opportunity can involve different combinations: stale plus visible, strong position plus weak CTR, or enough sessions plus weak engagement. Those signals overlap, have missingness, and can interact nonlinearly; a learned score may order borderline cases more effectively than independent thresholds. ML only wins if it improves held-out Precision@50, stays stable across clients and time, and still produces reason codes an editor can inspect. If those conditions fail, the fixed rule or a dashboard is the better solution. Even a successful model supports human review—it does not prove that a refresh will cause recovery.

In [5]:
stale_visible = lane_slice["days_since_last_update"].ge(180) & lane_slice["impressions_90d"].ge(500)
low_ctr_visible = (
    lane_slice["impressions_90d"].ge(500)
    & lane_slice["avg_position"].gt(0)
    & lane_slice["avg_position"].le(20)
    & lane_slice["ctr"].lt(0.5)  # rate columns use percentage points
)
weak_engagement = lane_slice["sessions_90d"].ge(30) & (
    (lane_slice["engagement_rate"].gt(0) & lane_slice["engagement_rate"].lt(30))
    | (lane_slice["scroll_rate"].gt(0) & lane_slice["scroll_rate"].lt(30))
)

reason_counts = pd.Series({
    "stale_visible": int(stale_visible.sum()),
    "low_ctr_visible": int(low_ctr_visible.sum()),
    "weak_engagement": int(weak_engagement.sum()),
    "low_ctr_and_weak_engagement": int((low_ctr_visible & weak_engagement).sum()),
    "any_reason_family": int((stale_visible | low_ctr_visible | weak_engagement).sum()),
})
print(reason_counts.to_string())
print("These overlapping candidate families motivate ranking, not automatic action.")


stale_visible                     17
low_ctr_visible                 9759
weak_engagement                 6508
low_ctr_and_weak_engagement     3082
any_reason_family              13190
These overlapping candidate families motivate ranking, not automatic action.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Saved under `work/notebooks/`; commit verification is completed with submission.